# Agente PLS: Colab QLoRA + AWS (respaldo)

Flujo paso a paso para generar resúmenes en lenguaje sencillo: instala dependencias, carga BioLaySumm, ejecuta una línea base, ajusta con QLoRA y empaqueta el adaptador para usarlo en Google Drive o AWS S3.

## Tabla de contenidos
- [1. Instalación y chequeo de GPU](#1-instalacion-y-chequeo-de-gpu)
- [2. Autenticación y montaje](#2-autenticacion-y-montaje)
- [3. Imports y configuración global](#3-imports-y-configuracion-global)
- [4. Carga de datos y DatasetDict](#4-carga-de-datos-y-datasetdict)
- [5. Vista previa y estadísticas rápidas](#5-vista-previa-y-estadisticas-rapidas)
- [6. Prompts y métricas de legibilidad](#6-prompts-y-metricas-de-legibilidad)
- [7. Configuración de LoRA/quantización](#7-configuracion-de-loraquantizacion)
- [8. Línea base zero-shot](#8-linea-base-zero-shot)
- [9. Utilidad de truncado](#9-utilidad-de-truncado)
- [10. Entrenamiento QLoRA opcional](#10-entrenamiento-qlora-opcional)
- [11. Carga del adaptador y evaluación](#11-carga-del-adaptador-y-evaluacion)
- [12. Empaquetado y carga a S3](#12-empaquetado-y-carga-a-s3)


## 1. Instalación y chequeo de GPU
Instala las dependencias principales (transformers, TRL, UnsloTh, métricas) y valida la GPU disponible en Colab.

In [ ]:
import os
import subprocess
import sys

if "google.colab" in sys.modules:
    subprocess.run(["nvidia-smi"], check=False)

requirements = [
    "accelerate>=0.34.0",
    "bitsandbytes==0.44.1",
    "transformers==4.44.2",
    "datasets==2.20.0",
    "evaluate==0.4.2",
    "peft==0.11.1",
    "trl==0.9.4",
    "bert-score==0.3.13",
    "textstat==0.7.4",
    "summac==0.1.0",
    "rouge-score==0.1.2",
    "boto3>=1.34.0",
    "unsloth>=2024.4.5",
]

subprocess.check_call([sys.executable, "-m", "pip", "install", "--quiet", "--upgrade"] + requirements)


## 2. Autenticación y montaje
Gestiona login en Hugging Face y, si se ejecuta en Colab, monta Google Drive para persistir artefactos.

In [ ]:
import os
from pathlib import Path
import sys

try:
    from huggingface_hub import login
except ImportError:
    login = None

HF_TOKEN = os.getenv("HF_TOKEN")

if login:
    if HF_TOKEN:
        login(token=HF_TOKEN, add_to_git_credential=True)
    elif "google.colab" in sys.modules:
        login()

if "google.colab" in sys.modules:
    from google.colab import drive

    drive.mount("/content/drive", force_remount=False)

PROJECT_ROOT = Path("/content/drive/MyDrive/pls_agent") if "google.colab" in sys.modules else Path.cwd()
PROJECT_ROOT.mkdir(parents=True, exist_ok=True)

ARTIFACT_DIR = PROJECT_ROOT / "artifacts"
ARTIFACT_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Artifacts:", ARTIFACT_DIR)


## 3. Imports y configuración global
Carga librerías de ML y métricas, fija la semilla, detecta dispositivo y prepara evaluadores como ROUGE y SummaC.

In [ ]:
import datetime
import json
import random
from typing import Any, Dict, List

import numpy as np
import pandas as pd
import torch
import textstat
import evaluate

from datasets import Dataset, DatasetDict
from transformers import (
    AutoModelForCausalLM,
    AutoTokenizer,
    BitsAndBytesConfig,
    TrainingArguments,
    pipeline,
)
from peft import LoraConfig, PeftModel, TaskType
from trl import SFTTrainer
from bert_score import score as bertscore_score
from unsloth import FastLanguageModel

try:
    from summac.model_summac import SummacEstimator
    HAVE_SUMMAC = True
except Exception as exc:
    print("⚠️ SummaC no disponible:", exc)
    HAVE_SUMMAC = False

try:
    from IPython.display import display
except Exception:
    display = None

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

rouge = evaluate.load("rouge")

if HAVE_SUMMAC:
    try:
        summac_estimator = SummacEstimator(model_name="vitc", backend="transformers", device=device)
    except Exception as exc:
        print("⚠️ Falló la carga de SummaC:", exc)
        HAVE_SUMMAC = False
        summac_estimator = None
else:
    summac_estimator = None

#BASE_MODEL_ID = "Qwen/Qwen2.5-1.5B-Instruct"
BASE_MODEL_ID = "unsloth/Qwen2.5-1.5B-Instruct-bnb-4bit"
MAX_SOURCE_TOKENS = 1024
MAX_TARGET_TOKENS = 256
MAX_SEQ_LENGTH = 1280
TRAIN_SAMPLES = 2000
VAL_SAMPLES = 200
SAMPLE_SIZE = 3
VAL_SPLIT_DEFAULT = "validation"


## 4. Carga de datos y DatasetDict
Localiza los CSV de pares PLS, construye DataFrames, estandariza columnas y genera un DatasetDict con splits de entrenamiento y validación.

In [ ]:
DATA_DIR = PROJECT_ROOT / "data"
if not DATA_DIR.exists():
    DATA_DIR = Path("data")

TRAIN_CSV = DATA_DIR / "pls_train_pairs.csv"
TEST_CSV = DATA_DIR / "pls_test_pairs.csv"

if not TRAIN_CSV.exists() or not TEST_CSV.exists():
    raise FileNotFoundError("No se encontraron los CSV de entrenamiento/validación en la carpeta data/")

train_df = pd.read_csv(TRAIN_CSV)
test_df = pd.read_csv(TEST_CSV)

def prepare_df(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()
    df["article"] = df["non_pls_text"].astype(str)
    df["summary"] = df["pls_text"].astype(str)
    df["title"] = df.get("base_id", "").fillna("")
    return df

train_df = prepare_df(train_df)
test_df = prepare_df(test_df)

train_ds = Dataset.from_pandas(train_df, preserve_index=False)
val_ds = Dataset.from_pandas(test_df, preserve_index=False)

ds_pairs = DatasetDict({"train": train_ds, "validation": val_ds})

print("Dataset sizes:", {name: len(ds_pairs[name]) for name in ds_pairs})
VAL_SPLIT = "validation"


## 5. Vista previa y estadísticas rápidas
Muestra ejemplos de los datos y calcula estadísticas de longitud para artículos y resúmenes en cada split.

In [ ]:
if "train" in ds_pairs:
    preview = ds_pairs["train"].select(range(min(3, len(ds_pairs["train"]))))
    df_preview = preview.to_pandas()[["title", "article", "summary"]]
    if display:
        display(df_preview)
    else:
        print(df_preview)


def describe_lengths(dataset, field: str, sample: int = 1024) -> Dict[str, float]:
    n = min(sample, len(dataset))
    subset = dataset.select(range(n))
    lengths = [len(text.split()) for text in subset[field]]
    return {
        "mean": float(np.mean(lengths)),
        "median": float(np.median(lengths)),
        "p90": float(np.percentile(lengths, 90)),
    }


for split in ds_pairs:
    stats_article = describe_lengths(ds_pairs[split], "article")
    stats_summary = describe_lengths(ds_pairs[split], "summary")
    print(f"{split.upper()} -> article words: {stats_article}, summary words: {stats_summary}")


## 6. Prompts y métricas de legibilidad
Define prompts de chunking y reducción estilo map-reduce y registra funciones de legibilidad objetivo.

In [ ]:
READABILITY_FUNCS = {
    "flesch_reading_ease": textstat.flesch_reading_ease,
    "flesch_kincaid_grade": textstat.flesch_kincaid_grade,
    "coleman_liau_index": textstat.coleman_liau_index,
    "gunning_fog": textstat.gunning_fog,
    "smog_index": textstat.smog_index,
    "dale_chall_score": textstat.dale_chall_readability_score,
}

CDC_TARGETS = {
    "flesch_reading_ease": ("min", 60.0),
    "flesch_kincaid_grade": ("max", 8.0),
    "coleman_liau_index": ("max", 8.0),
    "gunning_fog": ("max", 8.0),
    "smog_index": ("max", 8.0),
    "dale_chall_score": ("max", 8.0),
}

PROMPT_CHUNK = '''You are a medical writer translating clinical research into plain language for patients.
Follow the rules:
- Use everyday vocabulary and short sentences (<= 18 words).
- Define any unavoidable technical term in plain language.
- Highlight what was studied, key findings, and why they matter.
- Keep a neutral, factual tone.

Clinical text:
{chunk}

Plain-language micro-summary:'''

PROMPT_REDUCE = '''You are a medical writer. Merge the bullet summaries into one plain-language summary.
Requirements:
- 8-10 sentences, coherent and easy to understand.
- Cover study goal, methods, main results, safety signals, and limitations.
- Avoid hallucinations; stay faithful to the bullets.
- No jargon or acronyms unless defined.

Bullets:
{bullets}

Unified plain-language summary:'''


def chunk_by_tokens(text: str, tokenizer, max_input_tokens: int = MAX_SOURCE_TOKENS, overlap: int = 80) -> List[str]:
    ids = tokenizer(text, add_special_tokens=False)["input_ids"]
    if not ids:
        return [""]
    chunks = []
    start = 0
    while start < len(ids):
        end = min(start + max_input_tokens, len(ids))
        sub_ids = ids[start:end]
        chunks.append(tokenizer.decode(sub_ids, skip_special_tokens=True))
        if end == len(ids):
            break
        start = max(end - overlap, 0)
    return chunks


def readability_report(text: str) -> Dict[str, float]:
    scores = {}
    for name, func in READABILITY_FUNCS.items():
        try:
            scores[name] = float(func(text))
        except Exception:
            scores[name] = float("nan")
    return scores


def generate_summary(
    record: Dict[str, Any],
    generator,
    tokenizer,
    max_input_tokens: int = MAX_SOURCE_TOKENS,
    max_new_tokens: int = 320,
    temperature: float = 0.3,
) -> str:
    article = record.get("article", "")
    title = record.get("title", "")
    chunks = chunk_by_tokens(article, tokenizer, max_input_tokens=max_input_tokens)
    micro_summaries = []
    for chunk in chunks:
        prompt = PROMPT_CHUNK.format(title=title or "Untitled study", chunk=chunk)
        output = generator(
            prompt, max_new_tokens=max_new_tokens, do_sample=True, temperature=temperature
        )[0]["generated_text"]
        micro = output.split("Plain-language micro-summary:")[-1].strip()
        micro_summaries.append(micro)
    bullets = "\n- ".join(micro_summaries)
    final_prompt = PROMPT_REDUCE.format(bullets="- " + bullets if bullets else "-")
    final_output = generator(
        final_prompt, max_new_tokens=max_new_tokens, do_sample=False, temperature=0.0
    )[0]["generated_text"]
    summary = final_output.split("Unified plain-language summary:")[-1].strip()
    return summary


def evaluate_records(records: List[Dict[str, Any]], generator, tokenizer):
    predictions: List[str] = []
    references: List[str] = []
    sources: List[str] = []
    for rec in records:
        predictions.append(generate_summary(rec, generator, tokenizer))
        references.append(rec.get("summary", ""))
        sources.append(rec.get("article", ""))
    metrics: Dict[str, Any] = {}
    metrics["rouge"] = {
        k: float(v)
        for k, v in rouge.compute(
            predictions=predictions, references=references, use_stemmer=True
        ).items()
    }
    if predictions and references:
        P, R, F1 = bertscore_score(predictions, references, lang="en")
        metrics["bertscore"] = {
            "precision": float(P.mean()),
            "recall": float(R.mean()),
            "f1": float(F1.mean()),
        }
    else:
        metrics["bertscore"] = {}
    readability_all = [readability_report(text) for text in predictions]
    if readability_all:
        mean_scores = {
            k: float(np.nanmean([r[k] for r in readability_all])) for k in READABILITY_FUNCS
        }
        metrics["readability_mean"] = mean_scores
        cdc_alignment = {}
        cdc_pass_flags = []
        for metric, (mode, target) in CDC_TARGETS.items():
            value = mean_scores.get(metric, float("nan"))
            if np.isnan(value):
                meets = None
            else:
                meets = bool(value >= target) if mode == "min" else bool(value <= target)
                cdc_pass_flags.append(1 if meets else 0)
            cdc_alignment[metric] = {
                "value": value,
                "target": f">= {target}" if mode == "min" else f"<= {target}",
                "meets_target": bool(meets) if meets is not None else False,
            }
        metrics["readability_cdc_alignment"] = cdc_alignment
        metrics["readability_cdc_pass_rate"] = float(np.mean(cdc_pass_flags)) if cdc_pass_flags else float("nan")
    else:
        metrics["readability_mean"] = {}
        metrics["readability_cdc_alignment"] = {}
        metrics["readability_cdc_pass_rate"] = float("nan")
    if HAVE_SUMMAC and sources and predictions:
        try:
            factual = summac_estimator.predict(documents=sources, summaries=predictions)
            metrics["factuality_scores"] = [float(s) for s in factual["scores"]]
            metrics["factuality_mean"] = float(np.mean(metrics["factuality_scores"]))
        except Exception as exc:
            print("⚠️ SummaC falló durante la evaluación:", exc)
            metrics["factuality_scores"] = []
            metrics["factuality_mean"] = float("nan")
    else:
        metrics["factuality_scores"] = []
        metrics["factuality_mean"] = float("nan")
    metrics["prediction_length_mean"] = (
        float(np.mean([len(p.split()) for p in predictions])) if predictions else 0.0
    )
    metrics["reference_length_mean"] = (
        float(np.mean([len(r.split()) for r in references])) if references else 0.0
    )
    return metrics, predictions, references


## 7. Configuración de LoRA/quantización
Especifica parámetros de BitsAndBytes y LoRA para cargar el modelo base en 4 bits.

In [ ]:
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float16,
)

base_tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL_ID, use_fast=True, trust_remote_code=True)
if base_tokenizer.pad_token is None:
    base_tokenizer.pad_token = base_tokenizer.eos_token

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL_ID,
    device_map="auto",
    trust_remote_code=True,
    quantization_config=bnb_config,
)

baseline_generator = pipeline(
    task="text-generation",
    model=base_model,
    tokenizer=base_tokenizer,
)
baseline_generator.tokenizer.pad_token = (
    baseline_generator.tokenizer.pad_token or baseline_generator.tokenizer.eos_token
)
print("Baseline model loaded.")


## 8. Línea base zero-shot
Construye un muestreo de validación, genera predicciones iniciales y calcula métricas de referencia.

In [ ]:
eval_split = VAL_SPLIT if VAL_SPLIT in ds_pairs else next(iter(ds_pairs.keys()))
sample_ds = ds_pairs[eval_split].shuffle(SEED).select(range(min(SAMPLE_SIZE, len(ds_pairs[eval_split]))))
BASELINE_RECORDS = [sample_ds[i] for i in range(len(sample_ds))]

metrics_zero, baseline_preds, baseline_refs = evaluate_records(
    BASELINE_RECORDS, baseline_generator, base_tokenizer
)
print("Zero-shot metrics:", json.dumps(metrics_zero, indent=2))

baseline_df = pd.DataFrame(
    {
        "title": [rec.get("title", "") for rec in BASELINE_RECORDS],
        "reference_preview": [ref[:220] + "..." for ref in baseline_refs],
        "prediction_preview": [pred[:220] + "..." for pred in baseline_preds],
    }
)
if display:
    display(baseline_df)
else:
    print(baseline_df)


## 9. Utilidad de truncado
Incluye una función auxiliar para recortar textos a un máximo de tokens antes de la inferencia.

In [ ]:
def trim_text(text: str, tokenizer, max_tokens: int) -> str:
    ids = tokenizer(text, add_special_tokens=False, truncation=True, max_length=max_tokens)["input_ids"]
    return tokenizer.decode(ids, skip_special_tokens=True)


SYSTEM_PROMPT = (
    "You are a medical writer specialized in clinical trials. Produce factual, concise plain-language summaries "
    "understandable by patients and caregivers."
)
INSTRUCTION_TEMPLATE = """### Instrucción:
{system_prompt}

### Entrada:
{title_section}Texto clínico:
{article}

### Respuesta esperada:
"""


def format_example_for_sft(example):
    title = example.get("title") or ""
    title_section = f"Título: {title}\n\n" if title else ""
    article = trim_text(example["article"], base_tokenizer, MAX_SOURCE_TOKENS)
    prompt = INSTRUCTION_TEMPLATE.format(
        system_prompt=SYSTEM_PROMPT,
        title_section=title_section,
        article=article.strip(),
    )
    target = example["summary"].strip()
    return {"text": prompt + target + (base_tokenizer.eos_token or "")}


if "train" not in ds_pairs:
    raise ValueError("El dataset cargado no contiene división de entrenamiento.")

available_splits = [s for s in ds_pairs.keys() if s != "train"]
val_split_name = VAL_SPLIT if VAL_SPLIT in ds_pairs else (available_splits[0] if available_splits else "train")

train_subset = ds_pairs["train"].shuffle(SEED).select(range(min(TRAIN_SAMPLES, len(ds_pairs["train"]))))
val_subset = ds_pairs[val_split_name].shuffle(SEED).select(
    range(min(VAL_SAMPLES, len(ds_pairs[val_split_name])))
)

train_sft = train_subset.map(format_example_for_sft, remove_columns=train_subset.column_names)
val_sft = val_subset.map(format_example_for_sft, remove_columns=val_subset.column_names)

print("Train SFT samples:", len(train_sft))
print("Val SFT samples:", len(val_sft))


## 10. Entrenamiento QLoRA opcional
Habilita o deshabilita el fine-tuning con un flag y, si procede, entrena el modelo con SFTTrainer y guarda checkpoints.

In [ ]:
ADAPTER_DIR = ARTIFACT_DIR / "qlora_adapter"

DO_TRAIN = False  # cambia a True para ejecutar el fine-tuning (≈45-60 min con T4)

if DO_TRAIN:
    model_ft, tokenizer_ft = FastLanguageModel.from_pretrained(
        model_name=BASE_MODEL_ID,
        max_seq_length=MAX_SEQ_LENGTH,
        load_in_4bit=True,
        dtype=None,
        device_map="auto",
        trust_remote_code=True,
    )
    model_ft = FastLanguageModel.get_peft_model(
        model_ft,
        r=16,
        lora_alpha=32,
        target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
        lora_dropout=0.05,
        bias="none",
        task_type="CAUSAL_LM",
    )

    training_args = TrainingArguments(
        output_dir=str(ARTIFACT_DIR / "qlora_runs"),
        num_train_epochs=2,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=8,
        per_device_eval_batch_size=2,
        learning_rate=2e-4,
        logging_steps=50,
        evaluation_strategy="steps",
        eval_steps=200,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        bf16=torch.cuda.is_available(),
        fp16=not torch.cuda.is_available(),
        report_to="none",
        warmup_ratio=0.03,
        max_grad_norm=0.3,
    )

    trainer = SFTTrainer(
        model=model_ft,
        args=training_args,
        train_dataset=train_sft,
        eval_dataset=val_sft,
        tokenizer=tokenizer_ft,
        max_seq_length=MAX_SEQ_LENGTH,
        packing=True,
        dataset_text_field="text",
    )

    trainer.train()
    trainer.model.save_pretrained(ADAPTER_DIR)
    tokenizer_ft.save_pretrained(ADAPTER_DIR)
    print("Adapter guardado en", ADAPTER_DIR)
else:
    print("⚠️ Fine-tuning desactivado. Ajusta DO_TRAIN=True para lanzar el entrenamiento en Colab.")


## 11. Carga del adaptador y evaluación
Recupera el adaptador LoRA entrenado, evalúa con BERTScore/ROUGE y muestra algunas generaciones.

In [ ]:
if ADAPTER_DIR.exists():
    tuned_model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL_ID,
        device_map="auto",
        trust_remote_code=True,
        quantization_config=bnb_config,
    )
    tuned_model = PeftModel.from_pretrained(tuned_model, ADAPTER_DIR)
    tuned_generator = pipeline(
        task="text-generation",
        model=tuned_model,
        tokenizer=base_tokenizer,
    )
    tuned_generator.tokenizer.pad_token = (
        tuned_generator.tokenizer.pad_token or tuned_generator.tokenizer.eos_token
    )

    metrics_tuned, tuned_preds, tuned_refs = evaluate_records(
        BASELINE_RECORDS, tuned_generator, base_tokenizer
    )
    print("Tuned metrics:", json.dumps(metrics_tuned, indent=2))

    tuned_df = pd.DataFrame(
        {
            "title": [rec.get("title", "") for rec in BASELINE_RECORDS],
            "reference_preview": [ref[:220] + "..." for ref in tuned_refs],
            "prediction_preview": [pred[:220] + "..." for pred in tuned_preds],
        }
    )
    if display:
        display(tuned_df)
    else:
        print(tuned_df)
else:
    print("No se encontró adaptador entrenado en", ADAPTER_DIR)


## 12. Empaquetado y carga a S3
Comprime el adaptador final y lo sube a AWS S3 para usarlo en despliegues posteriores.

In [ ]:
import tarfile

import boto3

if ADAPTER_DIR.exists():
    tar_path = ARTIFACT_DIR / "qlora_adapter.tar.gz"
    with tarfile.open(tar_path, "w:gz") as tar:
        tar.add(ADAPTER_DIR, arcname="qlora_adapter")
    bucket = os.getenv("PLS_AWS_BUCKET", "your-s3-bucket")
    prefix = os.getenv("PLS_AWS_PREFIX", "pls-agent/")
    key = f"{prefix.rstrip('/')}/qlora_adapter_{datetime.datetime.utcnow().strftime('%Y%m%dT%H%M%SZ')}.tar.gz"
    if bucket.startswith("your-"):
        print("⚠️ Configura la variable de entorno PLS_AWS_BUCKET antes de subir a S3.")
        print("Archivo empaquetado localmente en", tar_path)
    else:
        s3 = boto3.client("s3")
        s3.upload_file(str(tar_path), bucket, key)
        print(f"Adaptador subido a s3://{bucket}/{key}")
else:
    print("No hay adaptador entrenado en", ADAPTER_DIR)
